# Aula 2: funções, comprehensions e erros

Na aula 1 vimos tipos, operadores, condicionais, listas, loops, tuplas, dicionários,
sets e conversões. Tudo isso era **dado**. Agora vem o que organiza o código:
empacotar lógica em funções, construir coleções de forma idiomática e lidar com o
que dá errado.

In [9]:
y = 0
taxa = 0

def x(var):
  z = 0
  taxa = 2
  print(f"{taxa=}")
  return taxa * var

print(f"{x(var=5)=}")
print(f"{x(var=10)=}")
print(f"{taxa=}")
print(z)

taxa=2
x(var=5)=10
taxa=2
x(var=10)=20
taxa=0


NameError: name 'z' is not defined

---
## 1. Funções

`def`, dois-pontos, corpo indentado. Sem declaração de tipo obrigatória e sem
chaves: a indentação delimita o corpo, como em todo o resto.

In [10]:
def calcular_total(valor, frete):
    return valor + frete

print(calcular_total(100, 20))

120


In [11]:
# Sem return explícito, a função devolve None
def cumprimentar(nome):
    print(f"Olá, {nome}!")

retorno = cumprimentar("Ana")
print(retorno)

Olá, Ana!
None


### Valores padrão e argumentos nomeados

Parâmetros com padrão vêm **depois** dos obrigatórios. Na chamada, você pode nomear
os argumentos: fica mais legível e libera você da ordem.

In [16]:
def calcular_total(valor, frete=0.0, desconto=0.0):
    return valor + frete - desconto

print(calcular_total(100))                            # só o obrigatório
print(calcular_total(100, 25))                        # posicional
print(calcular_total(100, desconto=10))               # nomeado, pulando o frete
print(calcular_total(desconto=10, valor=100))         # ordem não importa se nomear

100.0
125.0
90.0
90.0


In [19]:
# Isso levanta SyntaxError. Rode e leia a mensagem
def f(a=1, b):
    return a + b

SyntaxError: parameter without a default follows parameter with a default (3756105692.py, line 2)

### Retorno múltiplo

Python devolve uma tupla e você desempacota. É o mesmo unpacking da aula passada.

In [20]:
def estatisticas(valores):
    return min(valores), max(valores), sum(valores) / len(valores)

menor, maior, media = estatisticas([10, 20, 30, 40])
print(menor, maior, media)

# ou receba a tupla inteira
resultado = estatisticas([1, 2, 3])
print(resultado, type(resultado))

10 40 25.0
(1, 3, 2.0) <class 'tuple'>


### Docstring e type hints

Type hints são **opcionais** e não são verificados em tempo de execução: servem para
documentar e para a IDE te ajudar. Em código de projeto, use.

In [21]:
def aplicar_desconto(valor: float, percentual: float = 0.1) -> float:
    """Retorna o valor com o desconto percentual aplicado."""
    return valor * (1 - percentual)

print(aplicar_desconto(200))
print(aplicar_desconto(200, 0.25))
print(aplicar_desconto.__doc__)

180.0
150.0
Retorna o valor com o desconto percentual aplicado.


In [24]:
# O hint não é verificado: isto roda normalmente e devolve uma string
def dobrar(x: int) -> int:
    return x * 2

print(dobrar(21))
print(dobrar("ab"))     # o hint diz int, mas ninguém confere

42
abab


### Escopo

Variável criada dentro da função não existe fora. E atribuir a um nome dentro da
função cria uma variável **local**, mesmo que exista uma global com o mesmo nome.

In [25]:
def f():
    interna = 42
    return interna

print(f())
print(interna)   # descomente: NameError

42


NameError: name 'interna' is not defined

In [ ]:
contador = 0

def incrementar():
    contador = 10      # cria uma variável local, não toca na global
    return contador

print(incrementar())
print(contador)        # continua 0

> ⚠️ **Pegadinha clássica:** nunca use lista ou dicionário como valor padrão.
> O padrão é avaliado **uma única vez**, quando a função é definida, e fica
> compartilhado entre todas as chamadas.

In [27]:
def adicionar(item, lista=[]):     # ERRADO
    lista.append(item)
    return lista

print(adicionar("a"))
print(adicionar("b"))              # esperava ['b'], veio ['a', 'b']

['a']
['a', 'b']


In [28]:
def adicionar(item, lista=None):   # CERTO
    if lista is None:
        lista = []
    lista.append(item)
    return lista

print(adicionar("a"))
print(adicionar("b"))

['a']
['b']


**✏️ Exercícios rápidos**

1. `media(valores)` devolve a média da lista, e `0.0` quando a lista está vazia.
2. `formatar_moeda(valor, simbolo="R$")` devolve `"R$ 1234.50"`, com 2 casas.
3. `min_max(valores)` devolve o menor **e** o maior numa só chamada.
4. `contar_status(vendas)` recebe a lista de dicionários e devolve
   `{status: quantidade}`.
5. Sem rodar, preveja a saída de `misterio()` chamada três vezes. Depois confira.

In [ ]:
vendas = [
    {"vendedor": "Ana",   "valor": 3200.0, "status": "aprovada"},
    {"vendedor": "Bruno", "valor": 150.0,  "status": "aprovada"},
    {"vendedor": "Ana",   "valor": 890.0,  "status": "cancelada"},
]

def misterio(x, acc={}):
    acc[x] = acc.get(x, 0) + 1
    return acc

# 1)

# 2)

# 3)

# 4)

# 5)

---
## 2. Funções são objetos

Uma função pode ser guardada numa variável, passada como argumento e devolvida por
outra função. Isso destrava três ferramentas que você vai usar o tempo todo.

In [ ]:
def dobrar(x):
    return x * 2

f = dobrar              # sem parênteses: a função, não o resultado
print(f(21))
print(type(dobrar))

### `lambda`: função anônima de uma expressão

Serve para funções descartáveis, passadas como argumento. Se precisar de mais de uma
linha ou for reutilizar, escreva um `def` com nome.

In [ ]:
dobrar = lambda x: x * 2
print(dobrar(21))

soma = lambda a, b: a + b
print(soma(3, 4))

### `sorted(key=...)`

`key` recebe uma função aplicada a cada elemento para decidir a ordem. É a forma
padrão de ordenar por um critério.

In [ ]:
palavras = ["banana", "kiwi", "abacaxi", "uva"]

print(sorted(palavras))                      # alfabética
print(sorted(palavras, key=len))             # por tamanho
print(sorted(palavras, key=len, reverse=True))

In [ ]:
# ordenando dicionários por um campo
vendas = [
    {"vendedor": "Ana",   "valor": 3200.0},
    {"vendedor": "Bruno", "valor": 150.0},
    {"vendedor": "Carla", "valor": 890.0},
]

for v in sorted(vendas, key=lambda v: v["valor"], reverse=True):
    print(v["vendedor"], v["valor"])

In [ ]:
# ordenando um dict pelos valores
totais = {"Ana": 4090.0, "Bruno": 150.0, "Carla": 890.0}

print(sorted(totais.items(), key=lambda par: par[1], reverse=True))

### `zip`: percorrer coleções em paralelo

Junta elemento a elemento. Para quando a menor das coleções acabar.

In [ ]:
nomes = ["Ana", "Bruno", "Carla"]
notas = [4.5, 3.0, 5.0]

for nome, nota in zip(nomes, notas):
    print(f"{nome}: {nota}")

print(dict(zip(nomes, notas)))

**✏️ Exercícios rápidos**

1. Ordene `produtos` pelo preço, do mais caro para o mais barato.
2. Ordene `produtos` pelo nome, **ignorando maiúsculas/minúsculas**.
3. Com `zip`, monte `{produto: estoque}`.
4. Descubra qual vendedor tem o maior total em `totais`, usando `max` com `key`.

In [ ]:
produtos = [("Monitor", 900.0), ("teclado", 120.0), ("Mouse", 80.0)]
estoques = [4, 12, 30]
totais = {"Ana": 4090.0, "Bruno": 150.0, "Carla": 890.0}

# 1)

# 2)

# 3)

# 4)

---
## 3. Comprehensions

Forma idiomática de **construir uma coleção** a partir de outra. Os dois blocos
abaixo fazem exatamente a mesma coisa.

In [ ]:
nums = [1, 2, 3, 4, 5, 6]

dobros = []
for n in nums:
    dobros.append(n * 2)
print(dobros)

dobros = [n * 2 for n in nums]
print(dobros)

A estrutura é sempre:

```
[ expressão   for item in iterável   if condição ]
     ↑              ↑                     ↑
  o que sai      de onde vem         filtro (opcional)
```

In [ ]:
nums = [1, 2, 3, 4, 5, 6]

print([n for n in nums if n % 2 == 0])          # só filtra
print([n ** 2 for n in nums])                   # só transforma
print([n ** 2 for n in nums if n % 2 != 0])     # filtra e transforma

In [ ]:
# Sobre qualquer iterável, não só listas
vendas = [
    {"vendedor": "Ana",   "valor": 3200.0, "status": "aprovada"},
    {"vendedor": "Bruno", "valor": 150.0,  "status": "aprovada"},
    {"vendedor": "Carla", "valor": 890.0,  "status": "cancelada"},
]

print([v["vendedor"] for v in vendas if v["status"] == "aprovada"])
print(sum(v["valor"] for v in vendas if v["status"] == "aprovada"))

### Dict e set comprehension

Mesma sintaxe, mudando os delimitadores. Com `:` vira dict; sem `:`, dentro de
chaves, vira set.

In [ ]:
nomes = ["Ana", "Bruno", "Carla"]

print({nome: len(nome) for nome in nomes})              # dict

original = {"a": 1, "b": 2}
print({v: k for k, v in original.items()})              # invertendo um dict

palavras = ["gato", "cachorro", "gato", "peixe"]
print({p[0] for p in palavras})                         # set: iniciais distintas

### `if/else` na expressão

Quando o `if` vem **antes** do `for`, é o ternário: ele escolhe o valor, não filtra.

In [ ]:
nums = [1, 2, 3, 4]

print([n if n % 2 == 0 else 0 for n in nums])   # substitui  -> [0, 2, 0, 4]
print([n for n in nums if n % 2 == 0])          # filtra     -> [2, 4]

### Dois `for`: achatando listas aninhadas

Leia na mesma ordem em que escreveria os loops encaixados.

In [ ]:
matriz = [[1, 2], [3, 4], [5, 6]]

print([n for linha in matriz for n in linha])

> ⚠️ Comprehension serve para **construir uma coleção**. Se você só quer um efeito
> colateral (imprimir, salvar em disco), use `for` normal. E se a comprehension não
> couber confortavelmente em uma ou duas linhas, volte para o loop: legibilidade
> ganha de concisão.

**✏️ Exercícios rápidos**

1. Dos `nums`, monte a lista dos quadrados dos números maiores que 10.
2. Monte `{palavra: tamanho}` para as palavras com mais de 4 letras.
3. Monte o set das categorias distintas de `vendas`, usando `"não informada"`
   quando a chave faltar.
4. Achate `matriz` numa lista só e devolva-a ordenada, sem repetição.
5. Reescreva o loop da última célula como uma única comprehension.

In [ ]:
nums = [4, 8, 15, 16, 23, 42]
palavras = ["sol", "python", "mar", "dados", "ia"]
matriz = [[3, 1], [4, 1], [5, 9, 3]]
vendas = [
    {"vendedor": "Ana",   "valor": 3200.0, "categoria": "eletrônicos"},
    {"vendedor": "Bruno", "valor": 150.0},
    {"vendedor": "Carla", "valor": 890.0,  "categoria": "móveis"},
]

# loop do item 5
nomes_caros = []
for v in vendas:
    if v["valor"] > 500:
        nomes_caros.append(v["vendedor"].upper())
print(nomes_caros)

# 1)

# 2)

# 3)

# 4)

# 5)

---
## 4. try / except / finally

Em Python, exceção é mecanismo normal de fluxo, não último recurso. A cultura é
**EAFP** (*easier to ask forgiveness than permission*): tente fazer e trate o erro,
em vez de checar todas as condições antes.

In [ ]:
try:
    valor = float("abc")
except ValueError:
    print("não deu para converter")

In [ ]:
# LBYL (checar antes)  x  EAFP (tentar e tratar)
dados = {"valor": "150.0"}

# LBYL
if "valor" in dados and dados["valor"].replace(".", "").isdigit():
    print(float(dados["valor"]))

# EAFP: mais curto e cobre casos que você não previu
try:
    print(float(dados["valor"]))
except (KeyError, ValueError, TypeError):
    print("valor ausente ou inválido")

### Capture a exceção específica

`except Exception:` engole bug de verdade e transforma erro de programação em
comportamento silencioso. Capture o que você sabe tratar.

In [ ]:
def para_float(valor):
    """Converte para float, aceitando vírgula decimal. Devolve None se não der."""
    try:
        return float(str(valor).replace(",", "."))
    except (ValueError, TypeError):
        return None

for v in [15.5, "89.90", "45,50", None, "n/a"]:
    print(repr(v), "->", para_float(v))

### Exceções que você vai encontrar sempre

| Exceção | Quando |
|---|---|
| `ValueError` | tipo certo, valor inválido: `int("abc")` |
| `TypeError` | tipo errado: `"1" + 1` |
| `KeyError` | chave inexistente no dict |
| `IndexError` | índice fora da lista |
| `ZeroDivisionError` | divisão por zero |
| `FileNotFoundError` | arquivo não existe |

In [ ]:
pedido = {"id": 1, "valor": 150.0}
nums = [1, 2, 3]

for acao in ["chave", "indice", "divisao"]:
    try:
        if acao == "chave":
            pedido["categoria"]
        elif acao == "indice":
            nums[10]
        else:
            1 / 0
    except KeyError as e:
        print("KeyError:", e)
    except IndexError as e:
        print("IndexError:", e)
    except ZeroDivisionError as e:
        print("ZeroDivisionError:", e)

### `else` e `finally`

`else` roda quando **não** houve exceção. `finally` roda sempre, com ou sem erro,
com ou sem `return`. É onde vai a limpeza (fechar arquivo, conexão).

In [ ]:
def dividir(a, b):
    try:
        resultado = a / b
    except ZeroDivisionError:
        print("  divisão por zero")
        return None
    else:
        print("  deu certo")
        return resultado
    finally:
        print("  finally sempre roda")

print(dividir(10, 2))
print(dividir(10, 0))

### Levantando exceções

Quando a sua função recebe algo inválido, o certo é falhar alto, e não devolver um
valor esquisito que vai explodir três camadas adiante.

In [ ]:
def aplicar_desconto(valor: float, percentual: float) -> float:
    """Aplica um desconto entre 0 e 1 ao valor."""
    if not 0 <= percentual <= 1:
        raise ValueError(f"percentual fora do intervalo: {percentual}")
    return valor * (1 - percentual)

print(aplicar_desconto(100, 0.2))

try:
    aplicar_desconto(100, 1.5)
except ValueError as e:
    print("erro capturado:", e)

**✏️ Exercícios rápidos**

1. `parse_int(texto)` devolve o inteiro ou `None`, sem quebrar para `"abc"`, `None`
   ou `"12.5"`.
2. `pegar(lista, i, padrao=None)` devolve o elemento ou o padrão quando o índice
   não existe.
3. `soma_valores(vendas)` soma o campo `valor` **pulando** os registros em que ele
   está ausente ou é inválido, e devolve também quantos foram pulados.
4. `validar_venda(venda)` levanta `ValueError` se o valor for negativo e `KeyError`
   se faltar a chave `vendedor`. Teste os dois casos.

In [ ]:
vendas = [
    {"vendedor": "Ana",   "valor": 3200.0},
    {"vendedor": "Bruno", "valor": "150,50"},
    {"vendedor": "Carla"},
    {"vendedor": "Diego", "valor": "n/a"},
    {"vendedor": "Elena", "valor": None},
]

# 1)

# 2)

# 3)

# 4)

---
# 🏠 Para casa

## Parte A: Lab do notebook 1

Volte ao Lab de relatório de vendas, no fim do notebook da aula 1, e resolva as partes 1 a 7.

Agora com uma exigência a mais: **cada item deve virar uma função**, com nome claro,
docstring e type hints. Use comprehension onde ela deixar o código mais legível, e
`for` normal onde não deixar.

## Parte B: pipeline de limpeza

Os dados abaixo são o mesmo relatório de vendas, só que como chegam na vida real:
valor em formatos inconsistentes, chave faltando, `None`, texto solto, status escrito
de jeitos diferentes.

Este é, em miniatura, exatamente o trabalho que vamos fazer com Pandas mais para a
frente. Vale sentir a dor manualmente antes.

In [ ]:
vendas_sujas = [
    {"vendedor": "Ana",    "produto": "notebook", "categoria": "eletrônicos", "valor": 3200.00,   "mes": "jan", "status": "aprovada"},
    {"vendedor": "ana",    "produto": "cadeira",  "categoria": "móveis",      "valor": "890,00",  "mes": "JAN", "status": "Aprovada"},
    {"vendedor": "Bruno",  "produto": "mouse",    "categoria": None,          "valor": "150.00",  "mes": "jan", "status": "aprovada"},
    {"vendedor": "Bruno",  "produto": "mesa",     "categoria": "móveis",      "valor": None,      "mes": "fev", "status": "APROVADA"},
    {"vendedor": " Carla", "produto": "fone",                                 "valor": "R$ 250",  "mes": "fev", "status": "cancelada"},
    {"vendedor": "Carla",  "produto": "monitor",  "categoria": "eletrônicos", "valor": "n/a",     "mes": "fev", "status": "aprovada"},
    {"vendedor": "Diego",  "produto": "livro",    "categoria": "livros",      "valor": 120.00,    "mes": "mar", "status": "pendente"},
    {"vendedor": "Diego",  "produto": "camiseta", "categoria": "roupas",      "valor": -79.90,    "mes": "mar", "status": "aprovada"},
    {"vendedor": "Elena",  "produto": "notebook", "categoria": "eletrônicos", "valor": "3.100,00","mes": "mar", "status": "aprovada"},
]

print(f"{len(vendas_sujas)} registros")

**1.** `normalizar_texto(valor, padrao="não informada")` devolve o texto sem espaços
nas pontas e em minúsculas. Trata `None` e chave ausente devolvendo o padrão.

In [ ]:
# seu código aqui

**2.** `normalizar_valor(valor)` devolve `float` ou `None`. Precisa dar conta de:
número, `"890,00"`, `"150.00"`, `"3.100,00"` (milhar com ponto), `"R$ 250"`, `"n/a"`
e `None`.

_Dica: limpe a string antes de converter e use `try/except`._

In [ ]:
# seu código aqui


# normalizar_valor("3.100,00") -> 3100.0
# normalizar_valor("R$ 250")   -> 250.0
# normalizar_valor("n/a")      -> None

**3.** `limpar(vendas)` devolve **uma nova lista** (sem alterar a original) em que
cada registro tem `vendedor`, `produto`, `categoria`, `mes` e `status` normalizados,
e `valor` já como float. Descarte registros com valor inválido, ausente ou negativo.

Devolva também a lista dos registros descartados, para poder auditar o que se perdeu.

In [ ]:
# seu código aqui

**4.** `resumo(vendas)` recebe a lista **já limpa** e devolve:

```python
{
    "total_registros": 0,
    "faturamento": 0.0,
    "ticket_medio": 0.0,
    "por_vendedor": {},
    "por_categoria": {},
    "top_vendedor": "",
}
```

Considere apenas as vendas aprovadas. Levante `ValueError` se a lista vier vazia.

In [ ]:
# seu código aqui

**5. Desafio:** `agrupar_por(vendas, chave)` genérica, que recebe o nome de um campo e
devolve `{valor_do_campo: [registros]}`. Depois reescreva `por_vendedor` e
`por_categoria` do item 4 usando ela.

_Isto é, essencialmente, o `groupby` do Pandas escrito à mão._

In [ ]:
# seu código aqui

**6. Desafio:** imprima o relatório final alinhado em colunas: vendedor, número de
vendas, total e percentual do faturamento, ordenado do maior para o menor.

```
Vendedor     Vendas        Total      %
ana               2      4090.00   62.3%
bruno             1      1200.00   18.3%
```

In [ ]:
# seu código aqui

---
### Na próxima

Vamos empacotar essas funções num módulo de verdade: arquivos `.py`, imports,
ambiente e estrutura de projeto, para sair do notebook e virar código reaproveitável.